# Test Multi-Class Models on CIC-ToN-IoT (RDM-UQ) Dataset

This notebook tests the multi-class models trained on CICIDS2017 against the CIC-ToN-IoT dataset.

## Key Challenges:
1. **Different attack taxonomies**: CICIDS2017 vs RDM-UQ have different attack types
2. **Label mapping**: Need to map RDM-UQ attacks to CICIDS2017 categories
3. **Missing attack types**: Some attacks exist only in one dataset

## Attack Mapping Strategy:

### CICIDS2017 Classes (from training):
- BENIGN
- DDoS
- DoS (variants: GoldenEye, Hulk, Slowhttptest, slowloris)
- PortScan
- Bot
- FTP-Patator / SSH-Patator
- WebAttack (Brute Force, XSS, SQL Injection)
- Infiltration (removed in training)
- Heartbleed (removed in training)

### RDM-UQ Attack Types:
- Benign → **BENIGN**
- ddos → **DDoS**
- dos → **DoS**
- scanning → **PortScan**
- xss → **WebAttack**
- injection → **WebAttack** (SQL injection)
- password → **FTPPatator** (closest match - password attacks)
- backdoor → **Bot** (similar persistent access)
- mitm → **UNKNOWN** (no equivalent)
- ransomware → **UNKNOWN** (no equivalent)

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    classification_report
)
import time
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Load and Explore RDM-UQ Dataset

In [ ]:
# Load the CIC-ToN-IoT dataset
rdm_uq_path = "datasets/CSVs/dataset-rdm-uq/data/CIC-ToN-IoT.csv"
df_rdm = pd.read_csv(rdm_uq_path)

print(f"Dataset shape: {df_rdm.shape}")
print(f"\nAttack type distribution:")
print(df_rdm['Attack'].value_counts())
print(f"\nAttack percentages:")
print(df_rdm['Attack'].value_counts(normalize=True) * 100)

## 2. Attack Type Mapping

In [ ]:
# Define mapping from RDM-UQ to CICIDS2017 attack categories
attack_mapping = {
    'Benign': 'BENIGN',
    'ddos': 'DDoS',
    'dos': 'DoS',
    'scanning': 'PortScan',
    'xss': 'WebAttack',
    'injection': 'WebAttack',
    'password': 'FTPPatator',
    'backdoor': 'Bot',
    'mitm': 'UNKNOWN',  # No direct equivalent
    'ransomware': 'UNKNOWN'  # No direct equivalent
}

# Apply mapping
df_rdm['Label_Mapped'] = df_rdm['Attack'].map(attack_mapping)

print("Mapping Summary:")
print("="*60)
mapping_df = pd.DataFrame({
    'RDM-UQ Attack': list(attack_mapping.keys()),
    'CICIDS2017 Class': list(attack_mapping.values()),
    'Count': [df_rdm[df_rdm['Attack'] == k].shape[0] for k in attack_mapping.keys()]
})
print(mapping_df.to_string(index=False))
print("="*60)

print(f"\nMapped label distribution:")
print(df_rdm['Label_Mapped'].value_counts())

In [ ]:
# Visualize mapping
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Original RDM-UQ attacks
attack_counts = df_rdm['Attack'].value_counts()
colors_orig = ['green' if x == 'Benign' else 'red' for x in attack_counts.index]
attack_counts.plot(kind='bar', ax=axes[0], color=colors_orig)
axes[0].set_title('Original RDM-UQ Attack Types', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Attack Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Mapped CICIDS2017 classes
mapped_counts = df_rdm['Label_Mapped'].value_counts()
colors_mapped = ['green' if x == 'BENIGN' else ('orange' if x == 'UNKNOWN' else 'red') for x in mapped_counts.index]
mapped_counts.plot(kind='bar', ax=axes[1], color=colors_mapped)
axes[1].set_title('Mapped to CICIDS2017 Classes', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Filter Dataset for Testable Classes

We'll create two test sets:
1. **Full test set**: Including UNKNOWN classes (to see how models handle unseen attacks)
2. **Clean test set**: Only mapped classes that exist in training data

In [ ]:
# Full test set (with UNKNOWN)
df_full = df_rdm.copy()

# Clean test set (without UNKNOWN)
df_clean = df_rdm[df_rdm['Label_Mapped'] != 'UNKNOWN'].copy()

print(f"Full test set size: {len(df_full):,} samples")
print(f"Clean test set size: {len(df_clean):,} samples")
print(f"Removed {len(df_full) - len(df_clean):,} UNKNOWN samples ({(len(df_full) - len(df_clean))/len(df_full)*100:.2f}%)")

print(f"\nClean test set distribution:")
print(df_clean['Label_Mapped'].value_counts())

## 4. Preprocessing - Match Training Pipeline

In [ ]:
def preprocess_dataset(df, label_column='Label_Mapped'):
    """
    Apply the same preprocessing as training:
    1. Replace inf/-inf/nan with 0
    2. Normalize (log1p for positive skew, square for negative skew)
    3. Drop non-feature columns
    """
    df_copy = df.copy()
    
    # Replace infinities and NaN
    df_copy.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
    
    # Normalization
    numeric_columns = df_copy.select_dtypes(include=['number']).columns.tolist()
    for col in numeric_columns:
        if df_copy[col].skew() > 0 or col == "Src Port":
            df_copy[col] = np.log1p(df_copy[col].clip(lower=-0.99))
        elif df_copy[col].skew() < 0:
            df_copy[col] = df_copy[col] ** 2
    
    # Prepare features and labels
    X = df_copy.drop(["Flow ID", "Src IP", "Timestamp", "Dst IP", "Label", "Attack", label_column], axis=1)
    y = df_copy[label_column]
    
    return X, y

# Preprocess both datasets
print("Preprocessing clean test set...")
X_clean, y_clean = preprocess_dataset(df_clean)
print(f"✓ Clean set: X shape {X_clean.shape}, y shape {y_clean.shape}")

print("\nPreprocessing full test set...")
X_full, y_full = preprocess_dataset(df_full)
print(f"✓ Full set: X shape {X_full.shape}, y shape {y_full.shape}")

## 5. Load Trained Models and Preprocessing Objects

In [ ]:
# Paths to saved models
model_path = "../netflower/backend/src/ml_files/models/"
utils_path = "../netflower/backend/src/ml_files/utils/"

# Load scaler and PCA
with open(f"{utils_path}scaler.pkl", 'rb') as f:
    scaler = pickle.load(f)
print("✓ Loaded StandardScaler")

with open(f"{utils_path}pca.pkl", 'rb') as f:
    pca = pickle.load(f)
print(f"✓ Loaded PCA (n_components={pca.n_components_})")

# Load multi-class classification models
models = {}

# Decision Tree (multi-class)
with open(f"{model_path}dec_tree.pkl", 'rb') as f:
    models['Decision Tree'] = pickle.load(f)
print("✓ Loaded Decision Tree (multi-class)")

# KNN (multi-class)
with open(f"{model_path}knn.pkl", 'rb') as f:
    models['KNN'] = pickle.load(f)
print("✓ Loaded KNN (multi-class)")

# SVC (multi-class)
with open(f"{model_path}svc.pkl", 'rb') as f:
    models['SVC'] = pickle.load(f)
print("✓ Loaded SVC (multi-class)")

# Linear SVC (multi-class)
with open(f"{model_path}linear_svc.pkl", 'rb') as f:
    models['Linear SVC'] = pickle.load(f)
print("✓ Loaded Linear SVC (multi-class)")

print(f"\n✓ Loaded {len(models)} models successfully")

# Get class labels from models
if hasattr(models['Decision Tree'], 'classes_'):
    trained_classes = models['Decision Tree'].classes_
    print(f"\nTrained on classes: {list(trained_classes)}")

## 6. Apply Scaling and PCA Transformation

In [ ]:
# Transform clean dataset
print("Transforming clean test set...")
X_clean_scaled = scaler.transform(X_clean)
X_clean_pca = pca.transform(X_clean_scaled)
print(f"✓ Clean: {X_clean.shape} → {X_clean_pca.shape}")

# Transform full dataset
print("\nTransforming full test set...")
X_full_scaled = scaler.transform(X_full)
X_full_pca = pca.transform(X_full_scaled)
print(f"✓ Full: {X_full.shape} → {X_full_pca.shape}")

## 7. Evaluate Models on Clean Test Set

First, we test on clean data (only mapped classes that exist in training)

In [ ]:
# Evaluate all models on clean test set
results_clean = {}

for model_name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name} (Clean Test Set)")
    print(f"{'='*60}")
    
    # Measure prediction time
    start_time = time.time()
    y_pred = model.predict(X_clean_pca)
    prediction_time = time.time() - start_time
    
    # Calculate metrics
    accuracy = accuracy_score(y_clean, y_pred)
    precision = precision_score(y_clean, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_clean, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_clean, y_pred, average='weighted', zero_division=0)
    
    # Store results
    results_clean[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'prediction_time': prediction_time,
        'y_pred': y_pred
    }
    
    # Print results
    print(f"\nPrediction Time: {prediction_time:.4f} seconds")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Weighted Precision: {precision:.4f}")
    print(f"Weighted Recall: {recall:.4f}")
    print(f"Weighted F1-Score: {f1:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_clean, y_pred, zero_division=0))

print(f"\n{'='*60}")
print("✓ All models evaluated on clean test set")

## 8. Compare Model Performance (Clean Set)

In [ ]:
# Create comparison dataframe
results_clean_df = pd.DataFrame({
    'Model': list(results_clean.keys()),
    'Accuracy': [results_clean[m]['accuracy'] for m in results_clean],
    'Precision': [results_clean[m]['precision'] for m in results_clean],
    'Recall': [results_clean[m]['recall'] for m in results_clean],
    'F1-Score': [results_clean[m]['f1'] for m in results_clean],
    'Prediction Time (s)': [results_clean[m]['prediction_time'] for m in results_clean]
})

# Sort by F1-Score
results_clean_df = results_clean_df.sort_values('F1-Score', ascending=False)

print("\n" + "="*90)
print("MODEL PERFORMANCE COMPARISON - CLEAN TEST SET (Mapped Classes Only)")
print("="*90)
print(results_clean_df.to_string(index=False))
print("="*90)

In [ ]:
# Visualize metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = plt.cm.Set3(range(len(results_clean)))

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    
    values = results_clean_df[metric].values
    bars = ax.bar(results_clean_df['Model'], values, color=colors)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'{metric} Comparison (Clean Set)', fontsize=14, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('rdm_uq_multiclass_clean_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved comparison chart as 'rdm_uq_multiclass_clean_comparison.png'")

## 9. Confusion Matrices (Clean Set)

In [ ]:
# Get unique class labels
class_labels = sorted(y_clean.unique())

# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(20, 18))
axes = axes.ravel()

for idx, (model_name, result) in enumerate(results_clean.items()):
    cm = confusion_matrix(y_clean, result['y_pred'], labels=class_labels)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(ax=axes[idx], cmap='Blues', values_format='d', xticks_rotation=45)
    axes[idx].set_title(f'{model_name}\nF1-Score: {result["f1"]:.4f}', 
                       fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('rdm_uq_multiclass_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved confusion matrices as 'rdm_uq_multiclass_confusion_matrices.png'")

## 10. Per-Class Performance Analysis

In [ ]:
# Get best model
best_model_name = results_clean_df.iloc[0]['Model']
best_model_pred = results_clean[best_model_name]['y_pred']

print(f"Best performing model: {best_model_name}")
print(f"Weighted F1-Score: {results_clean[best_model_name]['f1']:.4f}")

# Calculate per-class metrics
class_performance = []
for class_label in class_labels:
    mask = y_clean == class_label
    true_labels = y_clean[mask]
    pred_labels = best_model_pred[mask]
    
    # Binary metrics for this class (one-vs-rest)
    true_binary = (y_clean == class_label).astype(int)
    pred_binary = (best_model_pred == class_label).astype(int)
    
    accuracy = accuracy_score(true_labels, pred_labels)
    precision = precision_score(true_binary, pred_binary, zero_division=0)
    recall = recall_score(true_binary, pred_binary, zero_division=0)
    f1 = f1_score(true_binary, pred_binary, zero_division=0)
    count = mask.sum()
    
    class_performance.append({
        'Class': class_label,
        'Count': count,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

class_perf_df = pd.DataFrame(class_performance)
class_perf_df = class_perf_df.sort_values('F1-Score', ascending=False)

print(f"\n{'='*90}")
print(f"PER-CLASS PERFORMANCE - {best_model_name}")
print(f"{'='*90}")
print(class_perf_df.to_string(index=False))
print(f"{'='*90}")

In [ ]:
# Visualize per-class performance
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(class_perf_df))
width = 0.2

bars1 = ax.bar(x - width*1.5, class_perf_df['Accuracy'], width, label='Accuracy', alpha=0.8)
bars2 = ax.bar(x - width*0.5, class_perf_df['Precision'], width, label='Precision', alpha=0.8)
bars3 = ax.bar(x + width*0.5, class_perf_df['Recall'], width, label='Recall', alpha=0.8)
bars4 = ax.bar(x + width*1.5, class_perf_df['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.set_xlabel('Attack Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'Per-Class Performance - {best_model_name}', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_perf_df['Class'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.savefig('rdm_uq_multiclass_per_class_performance.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved per-class performance chart")

## 11. Analyze Original RDM-UQ Attack Types

See how well models perform on the original attack types before mapping

In [ ]:
# Create dataframe with predictions and original attacks
analysis_df = pd.DataFrame({
    'Original_Attack': df_clean['Attack'],
    'Mapped_Class': y_clean,
    'Predicted_Class': best_model_pred
})

# Calculate accuracy per original attack type
original_attack_performance = []
for attack_type in sorted(df_clean['Attack'].unique()):
    mask = analysis_df['Original_Attack'] == attack_type
    true_labels = analysis_df.loc[mask, 'Mapped_Class']
    pred_labels = analysis_df.loc[mask, 'Predicted_Class']
    
    accuracy = accuracy_score(true_labels, pred_labels)
    count = mask.sum()
    mapped_to = df_clean[df_clean['Attack'] == attack_type]['Label_Mapped'].iloc[0]
    
    original_attack_performance.append({
        'Original Attack': attack_type,
        'Mapped To': mapped_to,
        'Count': count,
        'Accuracy': accuracy
    })

original_perf_df = pd.DataFrame(original_attack_performance)
original_perf_df = original_perf_df.sort_values('Accuracy', ascending=False)

print(f"\n{'='*80}")
print(f"PERFORMANCE ON ORIGINAL RDM-UQ ATTACK TYPES - {best_model_name}")
print(f"{'='*80}")
print(original_perf_df.to_string(index=False))
print(f"{'='*80}")

In [ ]:
# Visualize performance on original attacks
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['green' if x == 'Benign' else 'red' for x in original_perf_df['Original Attack']]
bars = ax.bar(original_perf_df['Original Attack'], original_perf_df['Accuracy'], color=colors, alpha=0.7)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}',
            ha='center', va='bottom', fontweight='bold', fontsize=9)

ax.set_xlabel('Original RDM-UQ Attack Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title(f'Detection Accuracy per Original Attack Type - {best_model_name}', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('rdm_uq_original_attack_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved original attack accuracy chart")

## 12. Summary and Insights

In [ ]:
print("\n" + "="*90)
print("SUMMARY: Multi-Class Model Testing on CIC-ToN-IoT Dataset")
print("="*90)

print(f"\nDataset Information:")
print(f"  - Total samples: {len(df_rdm):,}")
print(f"  - Testable samples (clean): {len(df_clean):,} ({len(df_clean)/len(df_rdm)*100:.2f}%)")
print(f"  - UNKNOWN samples filtered: {len(df_rdm) - len(df_clean):,} ({(len(df_rdm)-len(df_clean))/len(df_rdm)*100:.2f}%)")
print(f"  - Original attack types: {df_clean['Attack'].nunique()}")
print(f"  - Mapped classes: {len(class_labels)}")

print(f"\nBest Performing Model: {best_model_name}")
print(f"  - Accuracy: {results_clean[best_model_name]['accuracy']:.4f}")
print(f"  - Weighted Precision: {results_clean[best_model_name]['precision']:.4f}")
print(f"  - Weighted Recall: {results_clean[best_model_name]['recall']:.4f}")
print(f"  - Weighted F1-Score: {results_clean[best_model_name]['f1']:.4f}")
print(f"  - Prediction Time: {results_clean[best_model_name]['prediction_time']:.4f} seconds")

worst_model_name = results_clean_df.iloc[-1]['Model']
print(f"\nWorst Performing Model: {worst_model_name}")
print(f"  - Weighted F1-Score: {results_clean[worst_model_name]['f1']:.4f}")

best_class = class_perf_df.iloc[0]
print(f"\nBest Detected Class: {best_class['Class']}")
print(f"  - F1-Score: {best_class['F1-Score']:.4f}")

worst_class = class_perf_df.iloc[-1]
print(f"\nWorst Detected Class: {worst_class['Class']}")
print(f"  - F1-Score: {worst_class['F1-Score']:.4f}")

best_original = original_perf_df.iloc[0]
print(f"\nBest Detected Original Attack: {best_original['Original Attack']}")
print(f"  - Mapped to: {best_original['Mapped To']}")
print(f"  - Accuracy: {best_original['Accuracy']:.4f}")

worst_original = original_perf_df[original_perf_df['Original Attack'] != 'Benign'].iloc[-1]
print(f"\nWorst Detected Original Attack: {worst_original['Original Attack']}")
print(f"  - Mapped to: {worst_original['Mapped To']}")
print(f"  - Accuracy: {worst_original['Accuracy']:.4f}")

print("\nKey Findings:")
print("  1. Attack type mapping is crucial for cross-dataset evaluation")
print("  2. Some RDM-UQ attacks (mitm, ransomware) have no CICIDS2017 equivalent")
print("  3. Semantic similarity in mapping affects detection accuracy")
print("  4. Models show varying robustness to domain shift between datasets")
print("  5. Multi-class classification is more challenging than binary classification")

print("\nMapping Quality Insights:")
direct_maps = ['ddos→DDoS', 'dos→DoS', 'scanning→PortScan', 'xss→WebAttack']
indirect_maps = ['injection→WebAttack', 'password→FTPPatator', 'backdoor→Bot']
print(f"  - Direct mappings (high confidence): {len(direct_maps)}")
print(f"  - Indirect mappings (semantic similarity): {len(indirect_maps)}")
print(f"  - Unmappable attacks: 2 (mitm, ransomware)")

print("\n" + "="*90)

## Notes on Attack Mapping

### High-Confidence Mappings:
- **ddos → DDoS**: Direct match, distributed denial of service
- **dos → DoS**: Direct match, denial of service
- **scanning → PortScan**: Direct match, port/network scanning
- **xss → WebAttack**: XSS is a web attack type

### Semantic Similarity Mappings:
- **injection → WebAttack**: SQL/Command injection are web attacks (CICIDS2017 has SQL Injection in WebAttack)
- **password → FTPPatator**: Password attacks map to credential brute-force (FTP-Patator)
- **backdoor → Bot**: Both involve persistent unauthorized access/control

### Unmappable Attacks:
- **mitm**: Man-in-the-Middle attacks have no direct equivalent in CICIDS2017
- **ransomware**: Ransomware attacks are not present in CICIDS2017

### Challenges:
1. **Taxonomy differences**: Each dataset uses different attack categorization
2. **Granularity mismatch**: CICIDS2017 is more granular (e.g., DoS variants)
3. **Missing attack types**: Some modern attacks (ransomware) didn't exist in CICIDS2017
4. **Cross-domain generalization**: Models must generalize across IoT vs enterprise traffic